# Linked View: Airport Competition vs. Fare

**Norman's contribution to Example 2 — Monopolistic Pricing.**

Argument: airports served by fewer carriers have systematically higher average fares.

- **Map**: each airport dot colored by number of carriers (blue = competitive, red = monopoly)
- **Scatterplot**: avg fare vs carrier count — click a dot on the map to highlight it here

## 1. Setup & Data Prep

In [1]:
import pandas as pd
import numpy as np
import json
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML

pio.renderers.default = "plotly_mimetype+notebook_connected"

DATA      = "../../assets/data"
CLEAN_CSV = "../../data/cleandata/T_DB1B_MARKET_CLEAN.csv"

# Load airport carrier data (lat/lon + carrier shares)
with open(f"{DATA}/airport_carriers.json", encoding="utf-8") as f:
    ap_raw = json.load(f)

ap_df = pd.DataFrame([
    {
        "AIRPORT":       d["AIRPORT"],
        "LAT":           d["LAT"],
        "LON":           d["LON"],
        "TOTAL_PAX":     d["TOTAL_PAX"],
        "CARRIER_COUNT": len(d["carriers"]),
    }
    for d in ap_raw
])

# Compute avg fare per origin airport from clean CSV
fare_df = (
    pd.read_csv(CLEAN_CSV, usecols=["ORIGIN", "MARKET_FARE", "REPORTING_CARRIER"])
    .groupby("ORIGIN")
    .agg(AVG_FARE=("MARKET_FARE", "mean"))
    .reset_index()
    .rename(columns={"ORIGIN": "AIRPORT"})
)

airport = (
    ap_df
    .merge(fare_df, on="AIRPORT", how="inner")
    .query("LON > -130 and LON < -60 and LAT > 24 and LAT < 50")
    .reset_index(drop=True)
)

airport.to_json(f"{DATA}/airport_summary.json", orient="records", indent=2)
print(f"Airport summary: {len(airport)} airports")
print(airport[["AIRPORT","CARRIER_COUNT","AVG_FARE","TOTAL_PAX"]].describe())
airport.head()

Airport summary: 399 airports
       CARRIER_COUNT    AVG_FARE     TOTAL_PAX
count     399.000000  399.000000  3.990000e+02
mean        4.330827  285.297838  1.054730e+05
std         2.601860   59.064867  2.448796e+05
min         1.000000   65.021531  1.000000e+00
25%         2.000000  267.445553  1.197000e+03
50%         3.000000  289.929499  8.070000e+03
75%         8.000000  319.040142  5.637100e+04
max         8.000000  455.661368  1.391162e+06


,AIRPORT,LAT,LON,TOTAL_PAX,CARRIER_COUNT,AVG_FARE
0,ABE,40.652100,-75.440804,22109.0,5,277.829449
1,ABI,32.411301,-99.681900,8070.0,3,328.968875
2,ABQ,35.040199,-106.609001,183240.0,8,269.175378
3,ABR,45.449100,-98.421799,1862.0,2,387.611912
4,ABY,31.535500,-84.194504,2687.0,2,378.584124


## 2. Map — Carrier Count per Airport

In [2]:
BLUE_SCALE = [
    [0.0,  "#61a5c2"],
    [0.33, "#2c7da0"],
    [0.66, "#014f86"],
    [1.0,  "#013a63"],
]

fig_map = go.Figure(go.Scattergeo(
    lon=airport["LON"],
    lat=airport["LAT"],
    text=airport["AIRPORT"],
    customdata=airport[["AIRPORT", "CARRIER_COUNT", "AVG_FARE"]].values,
    mode="markers",
    marker=dict(
        size=(airport["TOTAL_PAX"] / airport["TOTAL_PAX"].max() * 22 + 5).clip(5, 27),
        color=airport["CARRIER_COUNT"],
        colorscale=BLUE_SCALE,
        reversescale=True,
        colorbar=dict(title="# Carriers", x=1.01),
        line=dict(width=0.5, color="white"),
        opacity=0.85,
    ),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Carriers: %{customdata[1]}<br>"
        "Avg Departing Fare: $%{customdata[2]:.0f}<br>"
        "<extra></extra>"
    ),
))

fig_map.update_layout(
    geo=dict(
        scope="usa",
        projection_type="albers usa",
        showland=True, landcolor="#f5f5f5",
        showlakes=True, lakecolor="#d0e8f5",
        showsubunits=True, subunitcolor="#ddd",
    ),
    height=420,
    margin=dict(l=0, r=60, t=30, b=0),
    paper_bgcolor="white",
    title=dict(text="Number of Carriers per Airport", x=0.5, font=dict(size=14)),
)

fig_map.show()

## 3. Scatterplot — Carrier Count vs. Avg Fare

In [3]:
from scipy.stats import pearsonr

r, _ = pearsonr(airport["CARRIER_COUNT"], airport["AVG_FARE"])

fig_scatter = go.Figure(go.Scatter(
    x=airport["CARRIER_COUNT"],
    y=airport["AVG_FARE"],
    mode="markers",
    text=airport["AIRPORT"],
    customdata=airport[["AIRPORT", "CARRIER_COUNT", "AVG_FARE", "TOTAL_PAX"]].values,
    marker=dict(
        size=8,
        color=airport["CARRIER_COUNT"],
        colorscale=BLUE_SCALE,
        reversescale=True,
        colorbar=dict(title="# Carriers", x=1.01),
        line=dict(width=0.5, color="white"),
        opacity=0.75,
    ),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Carriers: %{customdata[1]}<br>"
        "Avg Fare: $%{customdata[2]:.0f}<br>"
        "Passengers: %{customdata[3]:,.0f}<br>"
        "<extra></extra>"
    ),
))

# Trendline
m = np.polyfit(airport["CARRIER_COUNT"], airport["AVG_FARE"], 1)
x_line = np.linspace(airport["CARRIER_COUNT"].min(), airport["CARRIER_COUNT"].max(), 100)
fig_scatter.add_trace(go.Scatter(
    x=x_line, y=np.polyval(m, x_line),
    mode="lines", name=f"Trend (r = {r:.2f})",
    line=dict(color="#264653", dash="dash", width=1.5),
    hoverinfo="skip",
))

fig_scatter.update_layout(
    xaxis_title="Number of Carriers",
    yaxis_title="Average Fare (USD)",
    height=420,
    margin=dict(l=60, r=60, t=40, b=50),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=True,
    legend=dict(x=0.65, y=0.98),
    title=dict(text=f"Fewer Carriers \u2192 Higher Fares \u2002 r = {r:.2f}", x=0.5, font=dict(size=14)),
)
fig_scatter.update_xaxes(showgrid=True, gridcolor="#eee")
fig_scatter.update_yaxes(showgrid=True, gridcolor="#eee")

fig_scatter.show()

## 4. Linked View — Click Map to Highlight Scatterplot

In [4]:
map_html     = fig_map.to_html(full_html=False, include_plotlyjs=False, div_id='map-div')
scatter_html = fig_scatter.to_html(full_html=False, include_plotlyjs=False, div_id='scatter-div')

js = "<script>\n(function() {\n  function wireLink() {\n    var mapDiv     = document.getElementById('map-div');\n    var scatterDiv = document.getElementById('scatter-div');\n    if (!mapDiv || !scatterDiv) { setTimeout(wireLink, 300); return; }\n\n    mapDiv.on('plotly_click', function(data) {\n      var pt      = data.points[0];\n      var airport = pt.customdata[0];\n      var scatterData = scatterDiv.data[0];\n      var n = scatterData.text.length;\n      var colors = [], sizes = [];\n      for (var i = 0; i < n; i++) {\n        var match = (scatterData.text[i] === airport);\n        colors.push(match ? '#b2182b' : scatterData.marker.color[i]);\n        sizes.push(match ? 18 : 8);\n      }\n      Plotly.restyle(scatterDiv, {'marker.color': [colors], 'marker.size': [sizes]}, [0]);\n    });\n\n    mapDiv.on('plotly_doubleclick', function() {\n      var scatterData = scatterDiv.data[0];\n      var n = scatterData.text.length;\n      Plotly.restyle(scatterDiv, {\n        'marker.color': [scatterData.marker.color],\n        'marker.size':  [Array(n).fill(8)]\n      }, [0]);\n    });\n  }\n  wireLink();\n})();\n</script>"

HTML(
    '<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>'
    + '<p style="font-size:13px;color:#555;margin-bottom:8px;">'
    + 'Click an airport on the map to highlight it in the scatterplot. Double-click map to reset.</p>'
    + '<div style="display:flex;gap:12px;flex-wrap:wrap;">'
    + f'<div style="flex:1;min-width:340px">{map_html}</div>'
    + f'<div style="flex:1;min-width:340px">{scatter_html}</div>'
    + '</div>'
    + js
)

In [5]:
scatter_html = fig_scatter.to_html(full_html=True, include_plotlyjs='cdn')

with open("../../report/images/interactive_viz_2.html", "w", encoding="utf-8") as f:
    f.write(scatter_html)

print("Saved interactive_viz_2.html")

Saved interactive_viz_2.html
